[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch04-data-engineering/04-tu-danh-thuc-va-nhan.ipynb)

# Từ đánh thức và nhãn: đánh thức sai, đặc trưng âm thanh, đồng thuận và học chủ động

Chương 4 theo một ca xuyên suốt: hệ thống phát hiện từ khoá (keyword spotting, KWS) cho loa thông minh,
một mô hình nhỏ luôn bật, lắng nghe từ đánh thức (wake word) như "OK Google". Sổ tay này lấy ca đó làm
trục và đi qua phần chương nói về thu thập dữ liệu và gán nhãn: mỗi cửa sổ âm thanh phải bị từ chối chặt
đến đâu, ngân sách mua được bao nhiêu nhãn, đặc trưng âm thanh tốn bao nhiêu byte, và vì sao tiền nhãn lớn
hơn tiền compute hàng trăm lần.

**Bạn sẽ làm:**
1. tính tỉ lệ dương tính giả (false positive rate) mà một lần đánh thức sai mỗi tháng đòi hỏi (napkin math
   1.2);
2. chia ngân sách dữ liệu của napkin math 1.3, rồi so tiền nhãn với tiền compute;
3. tổng hợp một từ đánh thức, tính hệ số cepstral tần số mel (MFCC) bằng tay và đối chiếu với scipy;
4. đếm byte, phép tính và bộ nhớ của đặc trưng ở 8 và 16 kHz, 13 và 40 hệ số, trong ngân sách 16 KB;
5. huấn luyện một bộ phát hiện từ khoá nhỏ trên dữ liệu sạch, dữ liệu nhiễu và một nhóm giọng mà bộ
   sinh chưa từng tạo;
6. tính quy mô gán nhãn bằng tay, và đo mức đồng thuận giữa người gán nhãn (inter-annotator agreement) bằng
   kappa;
7. chạy học chủ động (active learning) trên MNIST, rồi xem độ tự tin của mô hình có đáng tin để gán nhãn sơ bộ
   (pre-annotation) không.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torchvision
from scipy import signal
from scipy.fft import dct as dct_scipy
from scipy.stats import binom
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
dat_hat_giong()
print("sẵn sàng")

## 1. Một lần đánh thức sai mỗi tháng

Napkin math 1.2: một loa thông minh luôn bật chỉ được đánh thức sai nhiều nhất một lần mỗi tháng. Mô hình
phân loại từng cửa sổ một giây, chạy suốt 24 giờ mỗi ngày. Vậy tỉ lệ dương tính giả (false positive rate,
FPR) trên mỗi cửa sổ phải nhỏ đến đâu?

**Dự đoán:** một mô hình đúng 99 % trên âm thanh không chứa từ khoá, tức FPR 1 %, sẽ đánh thức sai bao
nhiêu lần mỗi tháng? Vài lần, vài trăm lần, hay vài chục nghìn lần?

In [ ]:
GIAY_MOI_CUA_SO = 1                                     # cửa sổ một giây, không chồng lên nhau
CUA_SO_THANG = int(30 * 24 * 3600 / GIAY_MOI_CUA_SO)    # 30 ngày, 24 giờ mỗi ngày
theo_sach("số cửa sổ mỗi tháng", CUA_SO_THANG, sach=2592000, nguon="2.592.000 cửa sổ/tháng")

SAI_CHO_PHEP = sach(1, trich="1 tolerated false wake")
fpr = SAI_CHO_PHEP / CUA_SO_THANG
theo_sach("FPR mỗi cửa sổ", fpr, sach=3.9e-7, nguon="2.592.000 cửa sổ/tháng · 3,9 × 10⁻⁷")
theo_sach("tỉ lệ từ chối cửa sổ không chứa từ khoá (%)", 100 * (1 - fpr), sach=99.99996,
          nguon="3,9 × 10⁻⁷ · 99,99996 %")

FPR_1 = sach(1, nguon="1 % FPR") / 100
theo_sach("đánh thức sai mỗi tháng ở FPR 1 %", FPR_1 * CUA_SO_THANG, sach=25920,
          nguon="~25.920 lần đánh thức sai/tháng", tol=0.5)
print(f"tỉ lệ từ chối đủ chữ số: {100 * (1 - fpr):.7f} %")
print(f"FPR 1 % lớn gấp {FPR_1 / fpr:,.0f} lần mục tiêu, tức {math.log10(FPR_1 / fpr):.1f} bậc độ lớn")
print(f"số lần nhận sai mỗi giờ (FA/Hr) ở mục tiêu: {SAI_CHO_PHEP / (30 * 24):.4f}; ở FPR 1 %: {FPR_1 * 3600:.0f}")

In [ ]:
f = np.logspace(-8, -1, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(f, f * CUA_SO_THANG, color=MAU[0], lw=2)
ax.axhline(1, color="0.5", lw=1, ls="--")
ax.text(1.2e-8, 1.4, "một lần mỗi tháng", fontsize=8, color="0.3")
ax.plot([fpr, FPR_1], [1, FPR_1 * CUA_SO_THANG], "o", color=MAU[1], ms=8, zorder=3)
ax.annotate("mục tiêu của chương:\nFPR ≈ 3.9 × 10⁻⁷", (fpr, 1), xytext=(2.5e-6, 0.05), fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5"))
ax.annotate("FPR 1 % (đúng 99 %):\n25,920 lần mỗi tháng", (FPR_1, FPR_1 * CUA_SO_THANG), xytext=(3e-5, 3e4),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5"))
ax.set_xlabel("tỉ lệ dương tính giả trên mỗi cửa sổ một giây")
ax.set_ylabel("số lần đánh thức sai mỗi tháng")
ax.set_title("Napkin math 1.2: số cửa sổ mỗi tháng nhân với FPR")
plt.tight_layout()
plt.show()

"Đúng 99 %" nghe tốt, nhưng một tháng có 2,592,000 cửa sổ, nên 1 % thành 25,920 lần đánh thức sai, khoảng 36
lần mỗi giờ. Mục tiêu của chương nhỏ hơn thế hơn bốn bậc độ lớn. Vì vậy chương nói độ chính xác gộp là
không đủ: đánh giá phải báo số lần nhận sai mỗi giờ (false accepts per hour, FA/Hr) cùng với tỉ lệ bỏ sót đi
kèm.

Hai hệ quả mà napkin math không viết ra, cả hai là phép tính của sổ tay:

* **Người dùng thấy gì.** Giả định người dùng gọi từ đánh thức 10 lần mỗi ngày và mô hình bắt được 95 % số
  lần đó (tỉ lệ dương tính thật, true positive rate). Ô dưới đếm xem trong mọi lần loa thức dậy, bao nhiêu
  lần là đúng.
* **Chứng minh được mục tiêu cần bao nhiêu âm thanh.** Nếu thử $n$ cửa sổ không chứa từ khoá mà không thấy
  lần sai nào, thì với mức tin cậy 95 % ta chỉ kết luận được $\text{FPR} \le \ln(20)/n \approx 3/n$. Muốn kết
  luận được FPR dưới mục tiêu, $n$ phải đủ lớn.

In [ ]:
LAN_GOI_MOI_NGAY = 10   # giả định
TPR = 0.95              # giả định
dung = LAN_GOI_MOI_NGAY * 30 * TPR
for ten, ti_le in (("mục tiêu của chương", fpr), ("FPR 1 %", FPR_1)):
    sai = ti_le * CUA_SO_THANG
    print(f"{ten:>20}: {dung:.0f} lần thức đúng, {sai:>9,.0f} lần thức sai mỗi tháng"
          f" → {100 * dung / (dung + sai):5.1f} % số lần thức là đúng")

n_can = math.log(20) / fpr
print(f"\nđể kết luận FPR ≤ {fpr:.2e} với mức tin cậy 95 %: {n_can:,.0f} cửa sổ không chứa từ khoá liên tiếp không"
      f" một lần sai, tức {n_can * GIAY_MOI_CUA_SO / 86400:.0f} ngày âm thanh")

## 2. Ngân sách mua được bao nhiêu nhãn, và nhãn đắt hơn compute bao nhiêu

Napkin math 1.3 đặt các ràng buộc cho loa thông minh: độ chính xác 98 %, dưới một lần đánh thức sai mỗi
tháng, mô hình không quá 64 KB trong vùng luôn bật (always-on island), sáu tháng tới sản xuất, và $150K cho
toàn bộ kỹ thuật dữ liệu. Ngân sách chia ba phần: 60 % gán nhãn, 25 % lưu trữ và xử lý, 15 % quản trị và việc
khác. Một nhãn giá $0.10, cộng 20 % chi phí rà soát.

**Dự đoán:** tiền gán nhãn mua được nhiều hơn hay ít hơn mốc 1M ví dụ trong bảng 3?

In [ ]:
NGAN_SACH = sach(150e3, nguon="150K đô la · 90K (60 %)")
TI_LE = {"gán nhãn": sach(60, nguon="90K (60 %)"),
         "lưu trữ và xử lý": sach(25, nguon="37,5K (25 %)"),
         "quản trị và việc khác": sach(15, nguon="22,5K (15 %)")}
phan = {k: NGAN_SACH * v / 100 for k, v in TI_LE.items()}
assert sum(TI_LE.values()) == 100
theo_sach("tiền gán nhãn ($)", phan["gán nhãn"], sach=90e3, nguon="150K đô la · 90K (60 %)")
theo_sach("tiền lưu trữ và xử lý ($)", phan["lưu trữ và xử lý"], sach=37.5e3, nguon="37,5K (25 %)")
theo_sach("tiền quản trị ($)", phan["quản trị và việc khác"], sach=22.5e3, nguon="22,5K (15 %)")

GIA_NHAN = sach(0.10, nguon="0,10 đô la/nhãn")
RA_SOAT = sach(20, trich="20 percent review overhead") / 100
gia_mot_nhan = GIA_NHAN * (1 + RA_SOAT)
theo_sach("giá một nhãn kể cả rà soát ($)", gia_mot_nhan, sach=0.12, nguon="0,12 đô la/nhãn")
so_nhan = phan["gán nhãn"] / gia_mot_nhan
theo_sach("số nhãn mua được", so_nhan, sach=750e3, nguon="0,12 đô la/nhãn · 750K nhãn")

MOC = sach(1e6, nguon="1M vs 10M ví dụ")
print(f"\nbằng {so_nhan / MOC:.0%} mốc 1M; muốn đủ 1M nhãn thì giá một nhãn trước rà soát phải xuống"
      f" ${phan['gán nhãn'] / MOC / (1 + RA_SOAT):.3f}")
print("\nsố nhãn mua được bằng $90K (giá và mức rà soát ngoài $0.10 / 20 % là giả định):")
print("   giá một nhãn | rà soát 0 % |   20 % |   50 %")
for g in (0.05, 0.075, 0.10, 0.15, 0.20):
    print(f"   ${g:>11.3f} | " + " | ".join(f"{phan['gán nhãn'] / (g * (1 + r)) / 1e3:>5,.0f}K"
                                         for r in (0, 0.2, 0.5)))

Bước 3 của napkin math cộng thêm độ chính xác từ không gian thiết kế: một mô hình cơ sở minh hoạ đúng 90 %,
lấy mẫu 16 kHz thêm 2–4 %, tăng cường tổng hợp (synthetic augmentation) thêm 3–5 % độ bền. Bảng 3 ghi rõ các
hiệu ứng này **không được cộng thẳng**, vì chúng chồng lên nhau. Ô dưới cho thấy vì sao: cùng các mức tăng đó,
hai cách ghép khác nhau cho hai kết quả khác nhau, nên chỉ phép đo trên mô hình thật mới trả lời được mục tiêu
98 %.

In [ ]:
CO_SO = sach(90, trich="base-model accuracy of 90 percent")
TANG = {"lấy mẫu 16 kHz": (sach(2, trich="+2–4 percent accuracy"), sach(4, trich="+2–4 percent accuracy")),
        "tăng cường tổng hợp": (sach(3, trich="+3–5 percent robustness"), sach(5, trich="+3–5 percent robustness"))}
cong = [CO_SO + sum(v[i] for v in TANG.values()) for i in (0, 1)]
print(f"cộng thẳng: {cong[0]:.0f}–{cong[1]:.0f} %  (bảng 3 nói không được làm thế)")
# giả định khác: mỗi mức tăng là một phần lỗi còn lại của mô hình cơ sở, và các phần đó nhân với nhau
loi = 100 - CO_SO
con_lai = [loi * np.prod([1 - v[i] / loi for v in TANG.values()]) for i in (0, 1)]
print(f"nếu mỗi lựa chọn bớt một phần lỗi còn lại (giả định): {100 - con_lai[0]:.1f}–{100 - con_lai[1]:.1f} %")

Napkin math 1.3 dành 60 % ngân sách cho nhãn. Chương so tiền đó với tiền compute bằng bảng 5 và bảng 6, hai
bảng minh hoạ (bảng 5 ghi năm 2024): gán nhãn 1M ảnh bằng huy động cộng đồng (crowdsourcing) mất 2–4 tuần, huấn
luyện ResNet-50 trên ImageNet mất 4–6 giờ trên 8 GPU A100, và một lần tra kho đặc trưng (feature store) mất
1–10 ms. Một giờ GPU đám mây giá $2–4. Chương so $100K tiền nhãn với một lượt huấn luyện.

**Dự đoán:** $100K tiền nhãn bằng bao nhiêu lượt huấn luyện như thế?

In [ ]:
SO_GPU = sach(8, trich=r"8\(\times\) A100")
GIO_HL = (sach(4, nguon="4–6 giờ"), sach(6, nguon="4–6 giờ"))
GIA_GPU = (sach(2, nguon="2–4 đô la/giờ GPU"), sach(4, nguon="2–4 đô la/giờ GPU"))
luot_re, luot_dat = SO_GPU * GIO_HL[0] * GIA_GPU[0], SO_GPU * GIO_HL[1] * GIA_GPU[1]
theo_sach("một lượt huấn luyện, rẻ nhất ($)", luot_re, sach=64, nguon="64–192 đô la một lượt huấn luyện")
theo_sach("một lượt huấn luyện, đắt nhất ($)", luot_dat, sach=192, nguon="64–192 đô la một lượt huấn luyện")
TIEN_NHAN = sach(100e3, nguon="100K đô la nhãn")
theo_sach("tiền nhãn / lượt đắt nhất", TIEN_NHAN / luot_dat, sach=520.8, nguon="520,8×–1.562,5×")
theo_sach("tiền nhãn / lượt rẻ nhất", TIEN_NHAN / luot_re, sach=1562.5, nguon="520,8×–1.562,5×")

PHAN_CONG = sach(80, nguon="80 % công việc dồn vào 20 % đặc trưng")
PHAN_DUOI = sach(20, nguon="80 % công việc dồn vào 20 % đặc trưng")
gap_duoi = (PHAN_CONG / PHAN_DUOI) / ((100 - PHAN_CONG) / (100 - PHAN_DUOI))
print(f"\ntrong tiền nhãn, mỗi đặc trưng ở phần đuôi tốn công gấp {gap_duoi:.0f} lần một đặc trưng ở phần thân")

TUAN, GIO = 7 * 86400, 3600
KHOANG = {"gán nhãn 1M ảnh\n(huy động cộng đồng)": (sach(2, nguon="2–4 tuần") * TUAN, sach(4, nguon="2–4 tuần") * TUAN),
          "huấn luyện ResNet-50\n(8 GPU A100)": (GIO_HL[0] * GIO, GIO_HL[1] * GIO),
          "tra kho đặc trưng": (sach(1, nguon="1–10 ms") / 1000, sach(10, nguon="1–10 ms") / 1000)}
dai_nhat, ngan_nhat = KHOANG["gán nhãn 1M ảnh\n(huy động cộng đồng)"], KHOANG["tra kho đặc trưng"]
print(f"gán nhãn dài hơn huấn luyện {dai_nhat[0] / (GIO_HL[1] * GIO):.0f}–{dai_nhat[1] / (GIO_HL[0] * GIO):.0f} lần")
print(f"bảng 6 trải từ {math.log10(dai_nhat[0] / ngan_nhat[1]):.1f} đến"
      f" {math.log10(dai_nhat[1] / ngan_nhat[0]):.1f} bậc độ lớn, tuỳ đầu nào của mỗi khoảng")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
for i, (ten, (a, b)) in enumerate(KHOANG.items()):
    ax.plot([a, b], [i, i], color=MAU[i], lw=9, solid_capstyle="butt")
    chu = {0: "2–4 tuần", 1: "4–6 giờ", 2: "1–10 ms"}[i]
    ax.text(b * 3, i, chu, va="center", fontsize=9)
MOC_T = {"1 ms": 1e-3, "1 giây": 1, "1 phút": 60, "1 giờ": 3600, "1 ngày": 86400, "1 tháng": 30 * 86400}
for chu, x in MOC_T.items():
    ax.axvline(x, color="0.85", lw=0.8, zorder=0)
ax.set_xscale("log")
ax.set_xticks(list(MOC_T.values()), list(MOC_T.keys()))
ax.minorticks_off()
ax.annotate("", xy=(ngan_nhat[0], -0.55), xytext=(dai_nhat[1], -0.55),
            arrowprops=dict(arrowstyle="<->", color="0.4", lw=1))
ax.text(math.sqrt(ngan_nhat[0] * dai_nhat[1]), -0.7, "8.1–9.4 bậc độ lớn, tuỳ đầu khoảng", ha="center",
        fontsize=8, color="0.3")
ax.set_xlim(2e-4, 3e7)
ax.set_ylim(-1.0, 2.6)
ax.set_yticks(range(3), list(KHOANG.keys()))
ax.invert_yaxis()
ax.grid(False)
ax.set_title("Bảng 6 (minh hoạ): ba khâu trên một trục thời gian log")
plt.tight_layout()
plt.show()

Tiền nhãn gấp 520.8–1,562.5 lần một lượt huấn luyện trong kịch bản này, và thời gian gán nhãn dài hơn thời
gian huấn luyện từ 56 đến 168 lần. Đó là lý do chương gọi gán nhãn là điểm nghẽn, và vì sao phần 6 và 7 xoay
quanh câu hỏi làm sao mua ít nhãn hơn hoặc dùng nhãn đã mua tốt hơn. Ngay trong tiền nhãn, chương giả định 80 %
công việc dồn vào 20 % đặc trưng, tức phần đuôi gồm ca biên, lớp hiếm và ngoại lệ chất lượng: tính trên mỗi đặc
trưng, phần đuôi tốn công gấp 16 lần phần thân. Bảng 6 trải chín bậc độ lớn theo chú thích của nó; phép đếm ở
trên cho 8.1 đến 9.4 bậc tuỳ đầu khoảng được chọn.

## 3. Một từ đánh thức tổng hợp, và MFCC làm bằng tay

Sổ tay không tải âm thanh nào về. Nó tổng hợp một "từ đánh thức" ba âm tiết theo mô hình nguồn–bộ lọc: một
chuỗi xung ở tần số cơ bản $f_0$, tức cao độ của giọng, đi qua ba bộ cộng hưởng đặt ở ba formant của mỗi nguyên
âm. Các tần số formant gần với nguyên âm a, i, u, e, o và chỉ là **giả định** để âm có cấu trúc giống tiếng
nói. Từ dài 600 ms, nằm trong một cửa sổ một giây lấy mẫu ở 16 kHz.

Rồi sổ tay tính hai biểu diễn mà chương dùng cho âm thanh KWS: phổ thời gian–tần số (spectrogram) và hệ số
cepstral tần số mel. Các bước là cách tính MFCC thông dụng, với tham số là **giả định** của sổ tay: nhấn tần
cao với hệ số 0.97, cắt khung 25 ms cách nhau 10 ms, nhân cửa sổ Hann, biến đổi Fourier 512 điểm, gom năng
lượng vào 40 dải mel, lấy log, rồi biến đổi cosin rời rạc (DCT) và giữ 13 hệ số đầu.

In [ ]:
FS = 16000
NGUYEN_AM = {"a": (730, 1090, 2440), "i": (270, 2290, 3010), "u": (300, 870, 2240),
             "e": (530, 1840, 2480), "o": (570, 840, 2410)}  # ba formant (Hz), giả định
DAI_THONG = (80, 100, 140)                                 # độ rộng dải của mỗi formant (Hz), giả định
TU_DANH_THUC = "aiu"


def am_tiet(nguyen_am, n, f0_dau, f0_cuoi, co_giong, rng, fs=FS):
    """Một âm tiết: chuỗi xung theo cao độ, qua ba bộ cộng hưởng; co_giong nhân mọi formant."""
    f0 = np.linspace(f0_dau, f0_cuoi, n)
    xung = np.diff(np.floor(np.cumsum(f0) / fs), prepend=0.0)        # 1 ở đầu mỗi chu kỳ
    y = signal.lfilter([1.0], [1.0, -0.9], xung) + 0.01 * rng.standard_normal(n)
    for F, B in zip(NGUYEN_AM[nguyen_am], DAI_THONG):
        r, goc = math.exp(-math.pi * B / fs), 2 * math.pi * F * co_giong / fs
        y = signal.lfilter([1 - r], [1, -2 * r * math.cos(goc), r * r], y)
    y = y / np.sqrt(np.mean(y ** 2))
    return y * signal.windows.tukey(n, 0.4)


def tu(chuoi, dai_ms, f0, co_giong, rng, fs=FS):
    n = int(dai_ms / 1000 * fs) // len(chuoi)
    x = np.concatenate([am_tiet(v, n, f0 * (1.05 - 0.1 * j / len(chuoi)), f0 * (0.95 - 0.1 * j / len(chuoi)),
                                co_giong, rng, fs) for j, v in enumerate(chuoi)])
    return x / np.sqrt(np.mean(x ** 2))


def tron_nhieu(x, vung, snr_db, rng):
    """Cộng nhiễu trắng sao cho công suất của từ (trong `vung`) trên công suất nhiễu đúng bằng snr_db."""
    nhieu = rng.standard_normal(len(x))
    nhieu *= np.sqrt(np.mean(x[vung] ** 2) / 10 ** (snr_db / 10) / np.mean(nhieu ** 2))
    return x + nhieu, nhieu


rng = np.random.default_rng(7)
tu_mau = tu(TU_DANH_THUC, 600, 160, 1.0, rng)
LECH = int(0.2 * FS)                                  # từ bắt đầu ở 200 ms
VUNG = slice(LECH, LECH + len(tu_mau))
sach_am = np.zeros(FS)
sach_am[VUNG] = tu_mau
co_nhieu, nhieu_mau = tron_nhieu(sach_am, VUNG, 25, rng)
snr_that = 10 * math.log10(np.mean(sach_am[VUNG] ** 2) / np.mean(nhieu_mau ** 2))
assert abs(snr_that - 25) < 1e-9
print(f"một clip {len(co_nhieu) / FS:.0f} giây, {len(co_nhieu):,} mẫu; từ dài {len(tu_mau) / FS * 1000:.0f} ms;"
      f" SNR {snr_that:.1f} dB")

In [ ]:
def mel(f):
    return 2595 * np.log10(1 + f / 700)


def mel_nguoc(m):
    return 700 * (10 ** (m / 2595) - 1)


def bo_loc_mel(so_dai, nfft, fs):
    """40 bộ lọc tam giác, cách đều trên thang mel từ 0 tới fs/2."""
    diem = mel_nguoc(np.linspace(0, mel(fs / 2), so_dai + 2))
    f = np.fft.rfftfreq(nfft, 1 / fs)
    W = np.zeros((len(f), so_dai))
    for m in range(so_dai):
        a, b, c = diem[m:m + 3]
        W[:, m] = np.clip(np.minimum((f - a) / (b - a), (c - f) / (c - b)), 0, None)
    return W


def ma_tran_dct(k, n):
    """K hàng đầu của DCT-II chuẩn hoá trực giao, viết thành ma trận."""
    D = np.cos(np.pi / n * (np.arange(n)[None, :] + 0.5) * np.arange(k)[:, None]) * np.sqrt(2 / n)
    D[0] /= np.sqrt(2)
    return D


def cau_hinh(fs, k, so_dai=40):
    dai, buoc = int(0.025 * fs), int(0.010 * fs)
    nfft = 1 << (dai - 1).bit_length()                  # luỹ thừa của 2 nhỏ nhất không nhỏ hơn khung
    return dict(fs=fs, k=k, so_dai=so_dai, dai=dai, buoc=buoc, nfft=nfft, so_bin=nfft // 2 + 1,
                so_khung=1 + (fs - dai) // buoc, hann=signal.get_window("hann", dai),
                W=bo_loc_mel(so_dai, nfft, fs), D=ma_tran_dct(k, so_dai))


def khung(x, dai, buoc):
    so = 1 + (x.shape[-1] - dai) // buoc
    return x[..., np.arange(dai)[None, :] + buoc * np.arange(so)[:, None]]


def mfcc(x, c):
    """x: (số mẫu,) hoặc (số clip, số mẫu). Trả về MFCC, log-mel và phổ năng lượng."""
    if x.ndim == 2 and len(x) > 64:                     # chia lô để mảng khung không quá lớn
        phan_lo = [mfcc(x[i:i + 64], c) for i in range(0, len(x), 64)]
        return tuple(np.concatenate(p) for p in zip(*phan_lo))
    x = np.concatenate([x[..., :1], x[..., 1:] - 0.97 * x[..., :-1]], axis=-1)    # nhấn tần cao
    P = np.abs(np.fft.rfft(khung(x, c["dai"], c["buoc"]) * c["hann"], c["nfft"])) ** 2 / c["nfft"]
    log_mel = np.log(P @ c["W"] + 1e-10)
    return log_mel @ c["D"].T, log_mel, P


C16 = cau_hinh(16000, 13)
M, log_mel, P = mfcc(co_nhieu, C16)
print(f"phổ năng lượng {P.shape}, log-mel {log_mel.shape}, MFCC {M.shape}  (khung × giá trị)")

# đối chiếu: cùng khung và cửa sổ qua scipy.signal.stft, cùng DCT qua scipy.fft.dct
x_nhan = np.concatenate([co_nhieu[:1], co_nhieu[1:] - 0.97 * co_nhieu[:-1]])
_, _, Z = signal.stft(x_nhan, FS, window="hann", nperseg=C16["dai"], noverlap=C16["dai"] - C16["buoc"],
                      nfft=C16["nfft"], boundary=None, padded=False)
P_scipy = (np.abs(Z) * C16["hann"].sum()) ** 2 / C16["nfft"]      # stft chia cho tổng cửa sổ
assert P_scipy.shape == P.T.shape and np.allclose(P_scipy.T, P, rtol=1e-6, atol=1e-12)
assert np.allclose(dct_scipy(log_mel, type=2, norm="ortho", axis=-1)[:, :13], M)
print("✓ phổ năng lượng khớp scipy.signal.stft; ✓ MFCC khớp scipy.fft.dct")

In [ ]:
fig, o = plt.subplots(3, 2, figsize=(8.5, 7.5), gridspec_kw={"width_ratios": [40, 1]})
(a1, a2, a3), (c1, c2, c3) = o[:, 0], o[:, 1]
c1.axis("off")
t_ms = np.arange(FS) / FS * 1000
a1.plot(t_ms, co_nhieu, color="0.6", lw=0.5, label="có nhiễu, SNR 25 dB")
a1.plot(t_ms, sach_am, color=MAU[0], lw=0.6, label="từ đánh thức sạch")
a1.set_xlim(0, 1000)
a1.set_ylabel("biên độ")
a1.set_title("Dạng sóng: từ ba âm tiết a-i-u dài 600 ms trong cửa sổ một giây", fontsize=10)
a1.legend(loc="upper left", fontsize=8, frameon=False)
khung_ms = (np.arange(C16["so_khung"]) * C16["buoc"] + C16["dai"] / 2) / FS * 1000
anh = a2.imshow(log_mel.T, origin="lower", aspect="auto", cmap="magma",
                extent=[khung_ms[0], khung_ms[-1], 0.5, 40.5])
a2.set_ylabel("dải mel")
a2.set_title("Log năng lượng trong 40 dải mel (98 khung)", fontsize=10)
fig.colorbar(anh, cax=c2, label="log năng lượng")
M_z = (M - M.mean(0)) / M.std(0)                   # chuẩn hoá từng hệ số để cùng một thang màu
anh = a3.imshow(M_z.T, origin="lower", aspect="auto", cmap="coolwarm", vmin=-3, vmax=3,
                extent=[khung_ms[0], khung_ms[-1], -0.5, 12.5])
a3.set_yticks([0, 4, 8, 12])
a3.set_ylabel("hệ số MFCC")
a3.set_xlabel("thời gian (ms)")
a3.set_title("13 hệ số cepstral tần số mel: 98 × 13 giá trị cho cả giây", fontsize=10)
fig.colorbar(anh, cax=c3, label="độ lệch chuẩn,\ntheo từng hệ số")
for a in (a2, a3):
    a.grid(False)
    a.set_xlim(0, 1000)
plt.tight_layout()
plt.show()

Ở hình giữa, ba vùng sáng nối tiếp nhau là ba nguyên âm, mỗi nguyên âm sáng ở các dải mel của formant riêng
của nó. Hình dưới nén mỗi khung 40 giá trị xuống 13 bằng DCT: hệ số đầu theo độ to của khung, các hệ số sau theo
hình dáng của phổ.

Chương nêu các phép kiểm chất lượng riêng cho âm thanh KWS: theo mức tiếng ồn nền, với yêu cầu tỉ số tín
hiệu trên nhiễu (signal-to-noise ratio, SNR) trên 20 dB, và theo tốc độ nói, với yêu cầu từ đánh thức dài
500–800 ms. Quy trình thật không có bản sạch để so: nó phải ước lượng cả hai con số từ chính đoạn ghi âm. Sổ
tay dùng một cách ước lượng đơn giản, với hai tham số là **giả định**: năng lượng nền là phân vị 10 % của năng
lượng các khung, và khung có tiếng nói là khung cao hơn nền 6 dB.

**Dự đoán:** khi SNR thật giảm từ 30 dB xuống 0 dB, phép ước lượng SNR và phép đo độ dài hỏng theo kiểu nào?

In [ ]:
SNR_MIN = sach(20, trich="signal-to-noise ratio above 20 dB")
DAI_MIN = sach(500, nguon="500–800 ms độ dài mục tiêu của từ")
DAI_MAX = sach(800, nguon="500–800 ms độ dài mục tiêu của từ")


def uoc_luong(x, c=C16, phan_vi=10, nguong_db=6):
    E = np.mean(khung(x, c["dai"], c["buoc"]) ** 2, axis=-1)
    nen = np.percentile(E, phan_vi)
    noi = np.flatnonzero(E > nen * 10 ** (nguong_db / 10))
    if len(noi) == 0:
        return None, None
    snr = 10 * math.log10(max(E[noi].mean() - nen, 1e-30) / nen)
    return snr, ((noi[-1] - noi[0]) * c["buoc"] + c["dai"]) / c["fs"] * 1000


print(f"độ dài thật của từ: {len(tu_mau) / FS * 1000:.0f} ms")
print(" SNR thật | SNR ước lượng | độ dài đo được | qua kiểm SNR | qua kiểm độ dài")
for snr in (30, 25, 20, 15, 10, 5, 0):
    x, _ = tron_nhieu(sach_am, VUNG, snr, np.random.default_rng(3))
    s, d = uoc_luong(x)
    if s is None:
        print(f" {snr:>5} dB |  không thấy khung nào có tiếng nói")
        continue
    print(f" {snr:>5} dB | {s:>10.1f} dB | {d:>11.0f} ms | {'có' if s > SNR_MIN else 'không':>12} |"
          f" {'có' if DAI_MIN <= d <= DAI_MAX else 'không':>15}")

Đọc bảng từ trên xuống. Khi nhiễu lớn dần, hai đầu mỗi âm tiết chìm vào nền nên độ dài đo được ngắn lại, và phép
ước lượng SNR chỉ tính những khung đủ to nên nghiêng về phía cao. Ở SNR thấp nhất, phép dò có thể không tìm ra
khung nào để đo. Vì thế thứ tự kiểm tra có ý nghĩa: một clip phải qua kiểm SNR trước thì phép đo độ dài trên nó
mới đáng tin. Chương cũng nói rõ các phép sàng này không chứng minh một mẫu là hữu ích; chúng chỉ giảm khả năng
lỗi đã biết lọt vào khâu phát triển mô hình.

## 4. Đặc trưng tốn bao nhiêu byte, bao nhiêu phép tính, và có vừa 16 KB không

Bảng 3 định giá hai lựa chọn về đặc trưng. Lấy mẫu 16 kHz thay cho 8 kHz tốn 2 lần khối lượng xử lý đầu vào,
2 lần dung lượng âm thô và 2 lần kích thước đặc trưng. Dùng 40 hệ số thay cho 13 tốn 3 lần compute đặc trưng và
3 lần bộ nhớ đặc trưng. Bảng ghi rõ đây là đầu vào của một kịch bản minh hoạ, không phải số đo. Sổ tay đếm lại
từng thứ cho quy trình ở phần 3, ở cả hai tần số lấy mẫu và hai số hệ số.

**Dự đoán:** lên 16 kHz thì MFCC của một clip một giây có to gấp đôi không?

In [ ]:
CAU_HINH = {"8 kHz, 13 hệ số": cau_hinh(8000, 13), "16 kHz, 13 hệ số": C16, "16 kHz, 40 hệ số": cau_hinh(16000, 40)}
BYTE = 4                                     # float32, giả định
for ten, c in CAU_HINH.items():
    assert c["so_khung"] == 98
    c["byte_tho"] = c["fs"] * 2              # một giây, 16 bit, giả định
    c["byte_pho"] = c["so_khung"] * c["so_bin"] * BYTE
    c["byte_mfcc"] = c["so_khung"] * c["k"] * BYTE
    print(f"{ten}: âm thô {c['byte_tho']:>6,} B · phổ {c['so_khung']}×{c['so_bin']} = {c['byte_pho']:>7,} B"
          f" · MFCC {c['so_khung']}×{c['k']} = {c['byte_mfcc']:>6,} B")

c8, c16, c40 = CAU_HINH.values()
print()
theo_sach("âm thô, 16 kHz / 8 kHz", c16["byte_tho"] / c8["byte_tho"], sach=2, trich="2× raw-audio storage")
theo_sach("phổ thời gian–tần số, 16 kHz / 8 kHz", c16["byte_pho"] / c8["byte_pho"], sach=2,
          trich="2× feature size")
print(f"  MFCC, 16 kHz / 8 kHz: {c16['byte_mfcc'] / c8['byte_mfcc']:.2f}  (cùng 98 khung × 13 hệ số)")
theo_sach("MFCC, 40 / 13 hệ số", c40["byte_mfcc"] / c16["byte_mfcc"], sach=3, trich="3× feature memory")

Câu "2 lần kích thước đặc trưng" đúng với âm thô và với phổ thời gian–tần số, nhưng **không đúng với MFCC**:
số khung do bước 10 ms quyết định, số hệ số do ta chọn, và cả hai không phụ thuộc vào tần số lấy mẫu. Lên 16 kHz
chỉ làm mỗi dải mel trải rộng hơn trên trục tần số. Bảng 3 không nói nó định giá đặc trưng nào, nên đây không
phải lỗi của bảng, mà là lý do phải đếm lại cho đúng biểu diễn mình dùng.

Còn phép tính thì sao? Ô dưới đếm số phép tính cho một clip một giây, với giả định thông dụng là FFT $N$ điểm
tốn khoảng $5 N \log_2 N$ phép tính.

In [ ]:
def phep_tinh(c):
    moi_khung = {"nhấn tần cao và cửa sổ": 3 * c["dai"],
                 "FFT": 5 * c["nfft"] * math.log2(c["nfft"]),
                 "năng lượng": 3 * c["so_bin"],
                 "40 dải mel": 2 * c["so_bin"] * c["so_dai"],
                 "log và DCT": c["so_dai"] + 2 * c["so_dai"] * c["k"]}
    return {k: v * c["so_khung"] for k, v in moi_khung.items()}


for ten, c in CAU_HINH.items():
    c["phep"] = phep_tinh(c)
    print(f"{ten}: {sum(c['phep'].values()) / 1e6:.2f} triệu phép tính mỗi clip; FFT chiếm"
          f" {c['phep']['FFT'] / sum(c['phep'].values()):.0%}")
print()
theo_sach("phép tính trích đặc trưng, 16 kHz / 8 kHz", sum(c16["phep"].values()) / sum(c8["phep"].values()),
          sach=2, trich="2× input-processing workload")
print(f"  phép tính trích đặc trưng, 40 / 13 hệ số: {sum(c40['phep'].values()) / sum(c16['phep'].values()):.2f}")
print(f"  số giá trị một lớp đầu của mô hình phải đọc, 40 / 13 hệ số: {c40['k'] / c16['k']:.2f}")

Với cách đếm này, lên 16 kHz gần đúng gấp đôi khối lượng xử lý đầu vào, như bảng 3 nói. Nhưng 40 hệ số thay cho
13 chỉ thêm vài phần trăm cho khâu **trích** đặc trưng, vì FFT và 40 dải mel chiếm gần hết và không đổi. Thứ tăng
gấp ba là lượng giá trị mà mọi khâu phía sau phải lưu và đọc, nên con số "3 lần compute đặc trưng" của chương
khớp với phần tính trên đặc trưng, ví dụ lớp đầu của mô hình, chứ không khớp với phần trích. Chương không nói nó
đếm phần nào.

Ô dưới đo thời gian trích đặc trưng của ba cấu hình trên máy này. Thứ tự của ba con số có thể khác với thứ tự
của phép đếm, vì phép đếm bỏ qua chi phí cắt khung và gọi hàm.

In [ ]:
rng = np.random.default_rng(1)
lo_16 = np.stack([tron_nhieu(sach_am, VUNG, 20, rng)[0] for _ in range(64)])
lo_8 = signal.decimate(lo_16, 2, zero_phase=True, axis=-1)          # hạ xuống 8 kHz
assert lo_8.shape == (64, 8000)


def do_thoi_gian(x, c, lan=5):
    mfcc(x, c)                                        # chạy nóng một lần, không tính
    tot = math.inf
    for _ in range(lan):
        t0 = time.perf_counter()
        mfcc(x, c)
        tot = min(tot, time.perf_counter() - t0)
    return tot / len(x)


T_CLIP = {}
for ten, c in CAU_HINH.items():
    T_CLIP[ten] = do_thoi_gian(lo_8 if c["fs"] == 8000 else lo_16, c)
    do_tren_may(f"trích MFCC, {ten}", round(T_CLIP[ten] * 1e3, 3), "ms mỗi clip")

Chương thêm một ràng buộc khi triển khai: một ngân sách 16 KB cho trạng thái tiền xử lý, bên cạnh giới hạn 64 KB
của mô hình. Napkin math 1.3 nhắc rằng mô hình, trạng thái tiền xử lý, runtime và phần mạng phải được đo lượng bộ
nhớ chiếm dụng riêng. Ô dưới làm một bản kê như thế cho quy trình của sổ tay. Bản kê là **giả định**: mọi bảng và
bộ đệm (buffer) là float32 trừ khung âm thanh int16, bộ lọc mel chỉ lưu phần khác 0, và một bộ đệm vòng (ring
buffer) giữ MFCC của 98 khung gần nhất để mô hình đọc cả giây.

**Dự đoán:** cấu hình 40 hệ số vượt 16 KB vì đâu: vì FFT, vì bộ lọc mel, hay vì thứ khác?

In [ ]:
NGAN_SACH_TT = sach(16, trich="16 KB preprocessing-state budget") * 1000


def bo_nho(c, byte=BYTE, byte_vong=BYTE):
    return {"khung int16": c["dai"] * 2,
            "cửa sổ Hann": c["dai"] * byte,
            "bộ đệm FFT (số phức)": c["so_bin"] * 2 * byte,
            "phổ năng lượng": c["so_bin"] * byte,
            "bộ lọc mel, phần khác 0": np.count_nonzero(c["W"]) * byte + c["so_dai"] * 2 * 2,
            "log-mel": c["so_dai"] * byte,
            "ma trận DCT": c["k"] * c["so_dai"] * byte,
            "bộ đệm vòng 98 khung MFCC": c["so_khung"] * c["k"] * byte_vong}


for c in CAU_HINH.values():
    assert np.count_nonzero(c["W"]) <= 2 * c["so_bin"]      # mỗi bin FFT thuộc nhiều nhất hai tam giác
    c["bo_nho"] = bo_nho(c)
bang = {k: [c["bo_nho"][k] for c in CAU_HINH.values()] for k in c16["bo_nho"]}
print(f"{'':>27}" + "".join(f"{t:>18}" for t in CAU_HINH))
for k, v in bang.items():
    print(f"{k:>27}" + "".join(f"{x:>16,} B" for x in v))
tong = [sum(c["bo_nho"].values()) for c in CAU_HINH.values()]
print(f"{'tổng':>27}" + "".join(f"{x:>16,} B" for x in tong))
print(f"{'vừa 16 KB = 16,000 B?':>27}" + "".join(f"{'có' if x <= NGAN_SACH_TT else 'không':>18}" for x in tong))
print(f"{'vừa 16 KiB = 16,384 B?':>27}" + "".join(f"{'có' if x <= 16384 else 'không':>18}" for x in tong))

In [ ]:
NHOM = {"khung, cửa sổ, FFT, phổ": ["khung int16", "cửa sổ Hann", "bộ đệm FFT (số phức)", "phổ năng lượng"],
        "bộ lọc mel, log-mel, DCT": ["bộ lọc mel, phần khác 0", "log-mel", "ma trận DCT"],
        "bộ đệm vòng MFCC": ["bộ đệm vòng 98 khung MFCC"]}
fig, ax = plt.subplots(figsize=(8, 3.4))
ten_ch = list(CAU_HINH)
trai = np.zeros(len(ten_ch))
for (nhom, muc), mau in zip(NHOM.items(), MAU):
    rong = np.array([sum(c["bo_nho"][m] for m in muc) for c in CAU_HINH.values()]) / 1000
    ax.barh(ten_ch, rong, left=trai, color=mau, edgecolor="white", linewidth=2, label=nhom, height=0.6)
    trai += rong
ax.axvline(NGAN_SACH_TT / 1000, color="0.2", lw=1.2, ls="--")
ax.text(NGAN_SACH_TT / 1000 + 0.4, 0, "ngân sách 16 KB của chương", fontsize=8, va="center")
for i, x in enumerate(trai):
    gan_vach = NGAN_SACH_TT / 1000 - 1.5 < x <= NGAN_SACH_TT / 1000   # nhãn sẽ đè lên vạch 16 KB
    ax.text((NGAN_SACH_TT / 1000 if gan_vach else x) + 0.4, i, f"{x:.1f} KB", va="center", fontsize=8)
ax.set_xlim(0, 36)
ax.invert_yaxis()
ax.set_xlabel("trạng thái tiền xử lý (KB, bản kê giả định)")
ax.set_title("Bộ đệm vòng của MFCC là thứ tăng theo số hệ số")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

FFT và bộ lọc mel không đổi khi tăng số hệ số. Thứ tăng là ma trận DCT và nhất là bộ đệm vòng, vì nó giữ mọi hệ
số của cả giây. Với bản kê này, 13 hệ số vừa ngân sách, 40 hệ số thì không. Đây là lý do napkin math 1.3 chọn 13
hệ số "để giảm bộ nhớ trạng thái đặc trưng và compute".

Cuối cùng là quy mô của cả kho. Multilingual Spoken Words Corpus (MSWC) có 23.4 triệu ví dụ âm thanh, chiếm
748.8 GB âm thô. Chương nói trích đặc trưng theo từng file là song song hoàn toàn, nhưng tăng tốc đầu–cuối bị chặn
bởi băng thông của kho lưu trữ dùng chung. Bảng 8 cho kho lưu trữ đối tượng (object storage) 100–500 MB/s mỗi
kết nối.

In [ ]:
SO_CLIP = sach(23.4e6, nguon="23,4 triệu mẫu")
theo_sach("âm thô của cả kho (GB), một giây 16 kHz 16 bit mỗi clip", SO_CLIP * c16["byte_tho"] / 1e9,
          sach=748.8, nguon="23,4 triệu mẫu · 50 ngôn ngữ · 748,8 GB")
for ten, khoa in (("phổ thời gian–tần số", "byte_pho"), ("MFCC 13 hệ số", "byte_mfcc")):
    print(f"  cả kho dưới dạng {ten}, float32: {SO_CLIP * c16[khoa] / 1e9:,.1f} GB"
          f" ({c16[khoa] / c16['byte_tho']:.2f} lần âm thô)")

BW = (sach(100, nguon="100–500 MB/s"), sach(500, nguon="100–500 MB/s"))
theo_sach("đọc 748.8 GB ở 100 MB/s (giây)", SO_CLIP * c16["byte_tho"] / (BW[0] * 1e6), sach=7488,
          nguon="149,8 giây ở 5 GB/s vs 7.488 giây")
t1 = T_CLIP["16 kHz, 13 hệ số"]
for bw in BW:
    tran = bw * 1e6 / c16["byte_tho"]
    print(f"  một kết nối {bw} MB/s cấp tối đa {tran:,.0f} clip/giây")
    do_tren_may(f"số tiến trình trích đặc trưng làm đầy {bw} MB/s", math.ceil(tran * t1), "tiến trình")
do_tren_may("một tiến trình xử lý cả kho", round(SO_CLIP * t1 / 3600, 1), "giờ")

In [ ]:
so_tt = np.arange(1, 65)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(so_tt, so_tt / t1, color="0.6", lw=1, ls=":", label="nếu không có trần lưu trữ")
for bw, mau in zip(BW, MAU):
    tran = bw * 1e6 / c16["byte_tho"]
    ax.plot(so_tt, np.minimum(so_tt / t1, tran), color=mau, lw=2, label=f"kho đối tượng {bw} MB/s một kết nối")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks([1, 2, 4, 8, 16, 32, 64], ["1", "2", "4", "8", "16", "32", "64"])
ax.set_xlabel("số tiến trình trích đặc trưng")
ax.set_ylabel("clip mỗi giây")
ax.set_title("Song song hoàn toàn, cho tới khi gặp băng thông kho (đo trên máy này)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 5. Dữ liệu sạch, dữ liệu nhiễu, và giọng mà bộ sinh chưa từng tạo

Bảng 3 ghi hai lựa chọn về nguồn dữ liệu: dữ liệu có nhiễu thay cho dữ liệu sạch thêm 10–15 % độ chính xác trong
điều kiện thật nhưng tốn 3 lần chi phí thu thập, và tăng cường tổng hợp thay cho dữ liệu thật thêm 3–5 % độ bền
và rẻ hơn 10 lần. Phần ngộ nhận của chương thêm một cảnh báo: dữ liệu tổng hợp không thay được dữ liệu thật,
vì một hệ thống KWS chỉ học trên tiếng nói tổng hợp có thể bỏ sót giọng, tiếng ồn nền và cách phát âm mà bộ sinh
không tạo ra.

Sổ tay dựng một bộ phát hiện từ khoá nhỏ trên chính bộ sinh ở phần 3. Mỗi clip hoặc chứa từ đánh thức a-i-u, hoặc
một từ ba âm tiết khác ghép từ năm nguyên âm, với độ dài 500–800 ms, vị trí, cao độ và cỡ giọng ngẫu nhiên. Đặc
trưng là trung bình và độ lệch chuẩn theo thời gian của 13 MFCC, cộng trung bình của mỗi phần tư cửa sổ; mô hình
là hồi quy logistic. Mọi khoảng ngẫu nhiên dưới đây là **giả định**:

* **sạch:** SNR 30–40 dB; **nhiễu:** SNR 0–20 dB, đóng vai điều kiện thật; **trộn:** SNR 0–40 dB;
* **giọng quen:** cỡ giọng, tức hệ số nhân của mọi formant, 0.95–1.05 và cao độ 100–200 Hz;
* **giọng lạ:** cỡ giọng 1.12–1.22 và cao độ 220–300 Hz, gần với giọng trẻ em, một nhóm bộ sinh không tạo ra.

**Dự đoán:** mô hình học trên dữ liệu sạch làm tốt đến đâu trong điều kiện thật? Và cần bao nhiêu dữ liệu của
nhóm giọng lạ để kéo độ chính xác trên nhóm đó lên?

In [ ]:
SNR_SACH, SNR_NHIEU, SNR_TRON = (30, 40), (0, 20), (0, 40)
GIONG_QUEN = dict(co=(0.95, 1.05), f0=(100, 200))
GIONG_LA = dict(co=(1.12, 1.22), f0=(220, 300))
TU_KHAC = [a + b + c for a in NGUYEN_AM for b in NGUYEN_AM for c in NGUYEN_AM if a + b + c != TU_DANH_THUC]


def tao_tap(n, rng, snr, giong):
    """n clip một giây, nửa chứa từ đánh thức (nhãn 1), nửa chứa một từ khác (nhãn 0)."""
    X, y = np.zeros((n, FS)), np.arange(n) % 2
    for i in range(n):
        chuoi = TU_DANH_THUC if y[i] else TU_KHAC[rng.integers(len(TU_KHAC))]
        x = tu(chuoi, rng.uniform(500, 800), rng.uniform(*giong["f0"]), rng.uniform(*giong["co"]), rng)
        lech = rng.integers(0, FS - len(x))
        X[i, lech:lech + len(x)] = x
        X[i], _ = tron_nhieu(X[i], slice(lech, lech + len(x)), rng.uniform(*snr), rng)
    return X, y


def dac_trung(X):
    M = mfcc(X, C16)[0]
    tu_phan = [M[:, p].mean(1) for p in np.array_split(np.arange(M.shape[1]), 4)]
    return np.concatenate([M.mean(1), M.std(1)] + tu_phan, axis=1)


rng = np.random.default_rng(11)
TAP = {"huấn luyện sạch": tao_tap(1000, rng, SNR_SACH, GIONG_QUEN),
       "huấn luyện trộn": tao_tap(1000, rng, SNR_TRON, GIONG_QUEN),
       "giọng lạ để thêm": tao_tap(200, rng, SNR_TRON, GIONG_LA),
       "thử: sạch": tao_tap(400, rng, SNR_SACH, GIONG_QUEN),
       "thử: nhiễu": tao_tap(400, rng, SNR_NHIEU, GIONG_QUEN),
       "thử: giọng lạ, nhiễu": tao_tap(400, rng, SNR_NHIEU, GIONG_LA)}
DT = {k: (dac_trung(X), y) for k, (X, y) in TAP.items()}
for k, (F, y) in DT.items():
    print(f"{k:>22}: {len(y):>5} clip, {F.shape[1]} đặc trưng, {y.mean():.0%} chứa từ đánh thức")

In [ ]:
def hoc(F, y):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(F, y)


THU = ["thử: sạch", "thử: nhiễu", "thử: giọng lạ, nhiễu"]
KQ = {}
for ten in ("huấn luyện sạch", "huấn luyện trộn"):
    mo_hinh = hoc(*DT[ten])
    KQ[ten] = [mo_hinh.score(*DT[t]) for t in THU]
    print(f"{ten:>16}: " + " · ".join(f"{t[5:]} {100 * a:.1f} %" for t, a in zip(THU, KQ[ten])))

F_tron, y_tron = DT["huấn luyện trộn"]
F_la, y_la = DT["giọng lạ để thêm"]
TI_LE_LA = [0, 2, 5, 10, 20]                      # % tập huấn luyện thay bằng giọng lạ
TRON_LA = []
for p in TI_LE_LA:
    k = len(y_tron) * p // 100
    mo_hinh = hoc(np.vstack([F_tron[k:], F_la[:k]]), np.concatenate([y_tron[k:], y_la[:k]]))
    TRON_LA.append([mo_hinh.score(*DT["thử: giọng lạ, nhiễu"]), mo_hinh.score(*DT["thử: nhiễu"])])
    print(f"  {p:>2} % giọng lạ ({k:>3} clip): thử trên giọng lạ {100 * TRON_LA[-1][0]:.1f} %,"
          f" trên giọng quen {100 * TRON_LA[-1][1]:.1f} %")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.2, 1]})
x = np.arange(len(THU))
for j, (ten, mau) in enumerate(zip(KQ, MAU)):
    a1.bar(x + (j - 0.5) * 0.36, 100 * np.array(KQ[ten]), 0.34, color=mau, label=ten)
    for xi, v in zip(x, KQ[ten]):
        a1.text(xi + (j - 0.5) * 0.36, 100 * v + 1, f"{100 * v:.0f}", ha="center", fontsize=8)
a1.axhline(50, color="0.5", lw=1, ls=":", label="đoán bừa")
a1.set_xticks(x, ["sạch", "nhiễu\n(điều kiện thật)", "giọng lạ,\nnhiễu"])
a1.set_ylim(0, 110)
a1.set_ylabel("độ chính xác trên tập thử (%)")
a1.set_xlabel("tập thử")
a1.set_title("Học trên gì, thử trên gì (đo trên máy này)", fontsize=10)
a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=8, frameon=False)
TRON_LA = np.array(TRON_LA)
a2.plot(TI_LE_LA, 100 * TRON_LA[:, 0], "o-", color=MAU[2], lw=2, ms=6, label="thử trên giọng lạ")
a2.plot(TI_LE_LA, 100 * TRON_LA[:, 1], "s-", color=MAU[3], lw=2, ms=6, label="thử trên giọng quen")
a2.set_xticks(TI_LE_LA)
a2.set_xlabel("% tập huấn luyện trộn thay bằng giọng lạ")
a2.set_ylabel("độ chính xác trên tập thử nhiễu (%)")
a2.set_title("Thêm một ít dữ liệu của nhóm còn thiếu (đo trên máy này)", fontsize=10)
a2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Hình trái đặt cạnh nhau hai câu hỏi của bảng 3. Cột giữa hỏi dữ liệu có nhiễu mua được gì trong điều kiện thật:
một mô hình chỉ thấy âm sạch học cả những chi tiết mà nhiễu xoá mất. Cột phải hỏi điều mà ngộ nhận của chương
cảnh báo: cả hai mô hình chỉ học từ một bộ sinh, và bộ sinh không tạo ra nhóm giọng lạ. Hình phải thay một phần
nhỏ tập huấn luyện bằng dữ liệu của nhóm đó. Chương không đưa một tỉ lệ trộn tối ưu nào: nó khuyên kiểm định hỗn
hợp thật–tổng hợp trên dữ liệu đại diện cho lúc triển khai, đúng như việc tập thử giọng lạ làm ở đây. Các con số
của hình đến từ một bộ sinh đồ chơi và không so được với các mức tăng của bảng 3.

## 6. Gán nhãn ở quy mô 23.4 triệu clip, và đo mức đồng thuận

MSWC có 23.4 triệu ví dụ cho 340,000 từ khoá trong 50 ngôn ngữ. Chương tính: nếu mỗi nhãn chỉ mất 10 giây, gán
nhãn bằng tay cần khoảng 65,000 giờ, tức khoảng 32.5 người-năm làm toàn thời gian. Vì vậy hệ thống tự động bắt đầu
từ âm thanh của cả câu kèm bản chép lời, rồi dùng gióng hàng cưỡng bức (forced alignment) để ước lượng ranh giới
từng từ trong lời nói liên tục, và cắt mỗi từ vào một khung một giây.

In [ ]:
GIAY_NHAN = sach(10, nguon="10 giây mỗi nhãn")
gio_tay = SO_CLIP * GIAY_NHAN / 3600
theo_sach("giờ gán nhãn bằng tay", gio_tay, sach=65000, nguon="65.000 giờ ≈ 32,5 người-năm", tol=0.5)
GIO_MOT_NAM = 40 * 50                       # giả định: 40 giờ mỗi tuần, 50 tuần mỗi năm
theo_sach("người-năm", gio_tay / GIO_MOT_NAM, sach=32.5, nguon="65.000 giờ ≈ 32,5 người-năm", tol=0.05)
TU_KHOA = sach(340e3, trich="340,000 keywords in 50 languages")
print(f"trung bình {SO_CLIP / TU_KHOA:.0f} ví dụ mỗi từ khoá")
GIA_GIO = (sach(15, nguon="15–50 đô la/giờ rà soát"), sach(50, nguon="15–50 đô la/giờ rà soát"))
print(f"nếu trả theo giá giờ rà soát của bảng 5 (giả định áp cho việc gán nhãn):"
      f" ${gio_tay * GIA_GIO[0] / 1e6:.2f}M–${gio_tay * GIA_GIO[1] / 1e6:.2f}M,"
      f" gấp {gio_tay * GIA_GIO[0] / phan['gán nhãn']:.0f}–{gio_tay * GIA_GIO[1] / phan['gán nhãn']:.0f} lần"
      f" tiền gán nhãn của napkin math 1.3")

Khi đã có nhãn, câu hỏi tiếp theo là nhãn có nhất quán không. Mức đồng thuận giữa người gán nhãn (inter-annotator
agreement) được đo bằng kappa của Cohen, phương trình 2 của chương: $\kappa = (p_o - p_e)/(1 - p_e)$, với $p_o$ là
tỉ lệ hai người cùng nhãn và $p_e$ là tỉ lệ cùng nhãn do tình cờ. Ví dụ của chương là một dự án gán nhãn ảnh y
khoa: kappa 0.85 (đồng thuận gần như hoàn hảo) lúc đầu, giảm còn 0.72 (đồng thuận đáng kể) sau sáu tháng, khi người
gán nhãn mới vào mà không được đào tạo chuyên môn tương đương. Hiện tượng này là trôi chất lượng nhãn (label
quality drift).

Để đọc con số đó thành tỉ lệ nhãn sai, sổ tay dùng một mô hình **giả định**: bài toán hai lớp cân bằng, mỗi người
gán nhãn sai độc lập với cùng xác suất $e$. Khi đó $p_o = (1-e)^2 + e^2$, $p_e = 0.5$, nên $\kappa = (1-2e)^2$ và
$e = (1 - \sqrt{\kappa})/2$.

**Dự đoán:** từ 0.85 xuống 0.72, tỉ lệ nhãn sai của mỗi người tăng ít hơn hay nhiều hơn hai lần?

In [ ]:
def kappa(a, b):
    """Phương trình 2: (p_o − p_e) / (1 − p_e)."""
    a, b = np.asarray(a), np.asarray(b)
    p_o = np.mean(a == b)
    p_e = sum(np.mean(a == k) * np.mean(b == k) for k in np.union1d(a, b))
    return (p_o - p_e) / (1 - p_e)


def hai_nguoi(n, ti_le_duong, e, rng):
    """Nhãn thật, và nhãn của hai người, mỗi người lật nhãn thật với xác suất e, độc lập nhau."""
    that = (rng.random(n) < ti_le_duong).astype(int)
    a = np.where(rng.random(n) < e, 1 - that, that)
    b = np.where(rng.random(n) < e, 1 - that, that)
    return that, a, b


KAPPA = (sach(0.85, nguon="κ = 0,85 → 0,72"), sach(0.72, nguon="κ = 0,85 → 0,72"))
rng = np.random.default_rng(5)
E_SAI = {}
for k in KAPPA:
    e = (1 - math.sqrt(k)) / 2
    E_SAI[k] = e
    that, a, b = hai_nguoi(100000, 0.5, e, rng)
    assert abs(kappa(a, b) - cohen_kappa_score(a, b)) < 1e-12
    print(f"kappa {k}: mỗi người sai e = {100 * e:.2f} %; mô phỏng 100,000 mục cho kappa {kappa(a, b):.3f},"
          f" và hai người sai trung bình {50 * (np.mean(a != that) + np.mean(b != that)):.2f} %")
print(f"tỉ lệ sai tăng {E_SAI[KAPPA[1]] / E_SAI[KAPPA[0]]:.2f} lần; ✓ hàm kappa khớp sklearn")

Kappa giảm 0.13 nghe như một thay đổi nhỏ, nhưng dưới mô hình này nó nghĩa là tỉ lệ nhãn sai gần gấp đôi. Có hai
điều cần cẩn thận khi dùng phép đổi này.

* **Kappa phụ thuộc vào tỉ lệ lớp.** Trong KWS, hầu hết cửa sổ không chứa từ đánh thức. Cùng hai người gán nhãn
  với cùng $e$, khi lớp dương hiếm thì $p_e$ lớn lên và kappa tụt xuống, dù chất lượng từng người không đổi. Vì
  vậy một ngưỡng kappa chỉ so được giữa các đợt có cùng tỉ lệ lớp.
* **Bỏ phiếu đa số chỉ giúp khi lỗi độc lập.** Gán nhãn đồng thuận (consensus labeling) thu nhiều nhãn cho cùng
  một mục. Với $n$ người sai độc lập, đa số sai khi hơn nửa số người sai, một xác suất nhị thức. Một mục thật sự
  nhập nhằng, nơi mỗi người sai 40 % (giả định), thì bỏ phiếu gần như không cứu được: chương nói những mục đồng
  thuận thấp nên chuyển cho chuyên gia thay vì ép ra một nhãn. Và mỗi người thêm cho một mục chia số mục mà
  $90K của napkin math 1.3 mua được.

In [ ]:
def kappa_ly_thuyet(pi, e):
    q = pi * (1 - e) + (1 - pi) * e                  # tỉ lệ một người gán nhãn dương
    p_o, p_e = (1 - e) ** 2 + e ** 2, q ** 2 + (1 - q) ** 2
    return (p_o - p_e) / (1 - p_e)


def sai_da_so(e, n):
    return binom.sf(n // 2, n, e)                    # P(hơn nửa số người sai), n lẻ


for k, e in E_SAI.items():
    assert abs(kappa_ly_thuyet(0.5, e) - k) < 1e-12
pi = np.logspace(-2, math.log10(0.5), 100)
so_nguoi = np.array([1, 3, 5, 7, 9])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
for (k, e), mau in zip(E_SAI.items(), MAU):
    a1.semilogx(100 * pi, kappa_ly_thuyet(pi, e), color=mau, lw=2, label=f"mỗi người sai {100 * e:.1f} %")
    a1.plot([50], [k], "o", color=mau, ms=7)
    a1.annotate(f"κ = {k} của chương\n(lớp cân bằng)", (50, k), xytext=(14, 0.3) if k < 0.8 else (1.2, 0.9),
                fontsize=8, arrowprops=dict(arrowstyle="-", color="0.6"))
a1.set_xticks([1, 2, 5, 10, 20, 50], ["1", "2", "5", "10", "20", "50"])
a1.minorticks_off()
a1.set_xlabel("tỉ lệ mục thuộc lớp dương (%, thang log)")
a1.set_ylabel("kappa")
a1.set_ylim(0, 1)
a1.set_title("Cùng người gán nhãn, lớp dương càng hiếm thì kappa càng thấp", fontsize=10)
a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8, frameon=False)
for (ten, e), mau in zip({**{f"e = {100 * v:.1f} %": v for v in E_SAI.values()},
                          "mục nhập nhằng, e = 40 % (giả định)": 0.40}.items(), MAU):
    a2.semilogy(so_nguoi, 100 * sai_da_so(e, so_nguoi), "o-", color=mau, lw=2, ms=6, label=ten)
a2.set_xticks(so_nguoi)
a2.set_xlabel("số người gán nhãn mỗi mục")
a2.set_ylabel("nhãn đa số sai (%, thang log)")
a2.set_title("Bỏ phiếu đa số với lỗi độc lập", fontsize=10)
a2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
for n in (1, 3, 5):
    print(f"{n} người mỗi mục: $90K mua {so_nhan / n / 1e3:,.0f}K mục; nhãn đa số sai "
          + ", ".join(f"{100 * sai_da_so(e, n):.2f} %" for e in (*E_SAI.values(), 0.40)))

## 7. Học chủ động: nhãn nào đáng mua, và độ tự tin của mô hình có đáng tin không

Napkin math 1.8: một kho 10M ảnh, ngân sách gán nhãn $50K. Lấy mẫu ngẫu nhiên đạt 85 % với 100K nhãn, mục tiêu là
95 %. Một nhãn giá $0.50. Học chủ động (active learning) chọn những ví dụ khó để gán nhãn, nhưng muốn biết ví dụ
nào khó thì phải cho mô hình chấm điểm ứng viên, giá khoảng $0.01 một ảnh.

**Dự đoán:** chấm điểm cả 10M ảnh tốn bao nhiêu so với chính ngân sách?

In [ ]:
KHO = sach(10e6, nguon="10M ảnh · 50K đô la ngân sách")
NS = sach(50e3, nguon="10M ảnh · 50K đô la ngân sách")
GIA_NHAN_AL = sach(0.50, nguon="0,50 đô la/nhãn")
GIA_CHAM = sach(0.01, nguon="~0,01 đô la/ảnh suy luận", tol=0.005)
CAN_NGAU_NHIEN = sach(1e6, nguon="100K → 1M nhãn · 500K đô la")
theo_sach("ngẫu nhiên: tiền cho 1M nhãn ($)", CAN_NGAU_NHIEN * GIA_NHAN_AL, sach=500e3,
          nguon="100K → 1M nhãn · 500K đô la")
KHO_CAN = (sach(100e3, nguon="100K–200K ví dụ khó"), sach(200e3, nguon="100K–200K ví dụ khó"))
cham_het = KHO * GIA_CHAM
print(f"học chủ động, chỉ tiền nhãn: ${KHO_CAN[0] * GIA_NHAN_AL / 1e3:.0f}K–${KHO_CAN[1] * GIA_NHAN_AL / 1e3:.0f}K;"
      f" chấm điểm cả kho: ${cham_het / 1e3:.0f}K")
theo_sach("học chủ động kể cả chấm điểm, thấp ($)", KHO_CAN[0] * GIA_NHAN_AL + cham_het, sach=150e3,
          nguon="150K–200K đô la")
theo_sach("học chủ động kể cả chấm điểm, cao ($)", KHO_CAN[1] * GIA_NHAN_AL + cham_het, sach=200e3,
          nguon="150K–200K đô la")
TIEN_CHAM = NS * sach(10, trich="Spending 10 percent of this budget") / 100
theo_sach("10 % ngân sách cho suy luận ($)", TIEN_CHAM, sach=5e3, nguon="5K đô la chỉ chấm được 500K ứng viên")
theo_sach("số ứng viên chấm được", TIEN_CHAM / GIA_CHAM, sach=500e3, nguon="5K đô la chỉ chấm được 500K ứng viên")
theo_sach("số nhãn còn mua được", (NS - TIEN_CHAM) / GIA_NHAN_AL, sach=90e3, nguon="~90K nhãn", tol=500)

In [ ]:
cham = np.linspace(0, KHO, 400)
mua = np.maximum(NS - cham * GIA_CHAM, 0) / GIA_NHAN_AL
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhspan(KHO_CAN[0] / 1e3, KHO_CAN[1] / 1e3, color=MAU[3], alpha=0.25, lw=0)
ax.text(9.9, 150, "100K–200K ví dụ khó mà học chủ động\ncó thể cần (napkin math 1.8)", fontsize=8,
        ha="right", va="center")
ax.plot(cham / 1e6, mua / 1e3, color=MAU[0], lw=2)
ax.plot([TIEN_CHAM / GIA_CHAM / 1e6], [(NS - TIEN_CHAM) / GIA_NHAN_AL / 1e3], "o", color=MAU[1], ms=8, zorder=3)
ax.annotate("$5K chấm 500K ứng viên,\ncòn mua ~90K nhãn", (0.5, 90), xytext=(0.7, 15), fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5"))
ax.annotate("chấm 5M ảnh là hết\n$50K, không còn nhãn nào", (5, 0), xytext=(5.6, 40), fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5"))
ax.set_xlim(0, 10)
ax.set_ylim(0, 220)
ax.set_xlabel("số ảnh ứng viên được mô hình chấm điểm (triệu)")
ax.set_ylabel("số nhãn còn mua được (nghìn)")
ax.set_title("Ngân sách $50K: mỗi ảnh chấm điểm bớt đi 1/50 nhãn")
plt.tight_layout()
plt.show()

Đường xanh chỉ chạm mép dưới dải vàng ở điểm không chấm ảnh nào: với giá của napkin math, riêng tiền nhãn cho
100K ví dụ khó đã bằng cả ngân sách, nên mỗi đồng chi cho chấm điểm đẩy số nhãn xuống dưới mức cần. Chương kết luận học chủ động chỉ khả thi khi bể ứng viên được thu hẹp trước
khi chấm, khi giá suy luận giảm nhiều, hoặc khi compute được cấp ngân sách riêng.

Còn bản thân thuật toán thì sao? Chương nói học chủ động có thể đạt mục tiêu với ít mẫu hơn lấy mẫu ngẫu nhiên
"trong điều kiện thuận lợi". Sổ tay thử trên một bể 8,000 ảnh MNIST, rút gọn còn 40 thành phần chính bằng PCA
(không dùng nhãn), với hồi quy logistic. Cả hai cách bắt đầu từ cùng 100 nhãn ngẫu nhiên và mỗi vòng mua thêm
100 nhãn: một cách chọn ngẫu nhiên, cách kia là lấy mẫu theo độ bất định, chọn những ảnh có entropy dự đoán
(predictive entropy) cao nhất, phương trình 3 của chương: $H = -\sum_i p_i \log p_i$. Mỗi cách chạy với ba hạt
giống. Bên phải là đường cong học tập của lấy mẫu ngẫu nhiên tới cả bể 8,000 ảnh, để xem ngộ nhận "càng nhiều dữ
liệu càng tốt" của chương.

In [ ]:
tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
tap_thu = torchvision.datasets.MNIST(DATA, train=False, download=True)
rng = np.random.default_rng(0)
i_be, i_thu = rng.choice(len(tap), 8000, replace=False), rng.choice(len(tap_thu), 2000, replace=False)
X_be = tap.data.numpy()[i_be].reshape(-1, 784).astype(np.float32) / 255
y_be = tap.targets.numpy()[i_be]
X_thu = tap_thu.data.numpy()[i_thu].reshape(-1, 784).astype(np.float32) / 255
y_thu = tap_thu.targets.numpy()[i_thu]
pca = PCA(40, random_state=0).fit(X_be)
do_lech = pca.transform(X_be).std(0)
Z_be, Z_thu = pca.transform(X_be) / do_lech, pca.transform(X_thu) / do_lech


def mo_hinh_al(chi_so, C=1.0):
    return LogisticRegression(max_iter=3000, C=C).fit(Z_be[chi_so], y_be[chi_so])


def entropy(p):
    return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1)


N0_AL, BUOC_AL, SO_VONG_AL = 100, 100, 14          # 100 nhãn đầu, mỗi vòng mua thêm 100, 14 vòng


def chay_al(chien_luoc, hat, n0=N0_AL, buoc=BUOC_AL, so_vong=SO_VONG_AL):
    r = np.random.default_rng(hat)
    co_nhan = list(r.choice(len(Z_be), n0, replace=False))
    do_dung = []
    for vong in range(so_vong + 1):
        m = mo_hinh_al(co_nhan)
        do_dung.append(m.score(Z_thu, y_thu))
        if vong == so_vong:
            break
        con = np.setdiff1d(np.arange(len(Z_be)), co_nhan)
        if chien_luoc == "ngẫu nhiên":
            moi = r.choice(con, buoc, replace=False)
        else:
            moi = con[np.argsort(-entropy(m.predict_proba(Z_be[con])), kind="stable")[:buoc]]
        co_nhan += list(moi)
    return np.array(do_dung)


SO_NHAN_AL = N0_AL + BUOC_AL * np.arange(SO_VONG_AL + 1)
AL = {cl: np.array([chay_al(cl, h) for h in range(3)]) for cl in ("ngẫu nhiên", "theo độ bất định")}
moc = sorted({0, SO_VONG_AL // 3, 2 * SO_VONG_AL // 3, SO_VONG_AL})
for cl, a in AL.items():
    print(f"{cl:>17}: " + " · ".join(f"{SO_NHAN_AL[i]:,} nhãn {100 * a.mean(0)[i]:.1f} %" for i in moc))

thu_tu = np.random.default_rng(5).permutation(len(Z_be))
CO_HOC = np.array([125, 250, 500, 1000, 2000, 4000, 8000])
LOI = np.array([1 - mo_hinh_al(thu_tu[:n]).score(Z_thu, y_thu) for n in CO_HOC])
for n, a, b in zip(CO_HOC[1:], LOI[:-1], LOI[1:]):
    print(f"  {n // 2:>5} → {n:>5} nhãn ngẫu nhiên: lỗi giảm {100 * (a - b):.1f} điểm phần trăm")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
for (cl, a), mau in zip(AL.items(), MAU):
    a1.fill_between(SO_NHAN_AL, 100 * a.min(0), 100 * a.max(0), color=mau, alpha=0.2, lw=0)
    a1.plot(SO_NHAN_AL, 100 * a.mean(0), "o-", color=mau, lw=2, ms=4, label=cl)
a1.set_xlabel("số nhãn đã mua")
a1.set_ylabel("độ chính xác trên 2,000 ảnh thử (%)")
a1.set_title("Ba hạt giống, dải là thấp nhất–cao nhất (đo trên máy này)", fontsize=10)
a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8, frameon=False)
a2.loglog(CO_HOC, 100 * LOI, "o-", color=MAU[0], lw=2, ms=6)
for n, l in zip(CO_HOC, LOI):
    a2.text(n * 1.08, 100 * l * 1.04, f"{100 * l:.1f}", fontsize=8)
a2.set_xticks(CO_HOC, [f"{n:,}" for n in CO_HOC])
a2.set_yticks([10, 15, 20, 30], ["10", "15", "20", "30"])
a2.minorticks_off()
a2.set_xlabel("số nhãn ngẫu nhiên (thang log)")
a2.set_ylabel("tỉ lệ lỗi trên ảnh thử (%, thang log)")
a2.set_title("Mỗi lần gấp đôi dữ liệu mua được ít hơn (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Khoảng cách giữa hai đường ở hình trái là thứ học chủ động mua được trên bể này. Nó phụ thuộc vào dữ liệu, vào mô
hình và vào cỡ mỗi vòng mua, và có thể hẹp, bằng 0, hay đổi chiều trên một bài toán khác. Đó là lý do chương chỉ
hứa lợi ích "trong điều kiện thuận lợi". Hình phải là phần ngộ nhận đầu tiên của chương: qua một ngưỡng, dữ liệu
thêm cho lợi ích giảm dần, và mất mát trên tập kiểm tra thường theo một luật lũy thừa của cỡ tập dữ liệu. Ở đây
đường còn phẳng dần về một mức sàn, vì một mô hình tuyến tính trên 40 thành phần không học được hết bài toán dù
có bao nhiêu nhãn.

Chương còn nói entropy dự đoán chỉ là tín hiệu bất định có ích khi xác suất của mô hình đã được hiệu chuẩn
(calibration). Điều đó càng quan trọng khi mô hình gán nhãn sơ bộ (pre-annotation) cho người rà soát: nếu AI báo
95 % tự tin nhưng chỉ đúng 75 % ở mức tự tin đó, nhãn sơ bộ sẽ dẫn người rà soát đi sai. Phép đo chuẩn là biểu
đồ hiệu chuẩn: chia dự đoán theo độ tự tin, rồi so độ tự tin trung bình với tỉ lệ đúng trong mỗi khoảng. Sai số
hiệu chuẩn kỳ vọng (expected calibration error, ECE) là trung bình có trọng số của khoảng cách đó.

**Dự đoán:** nới điều chuẩn của hồi quy logistic (tăng `C` từ 1 lên 100) làm mô hình tự tin hơn. Nó có đúng
nhiều hơn ở những dự đoán tự tin không?

In [ ]:
TU_TIN = sach(95, nguon="95 % tự tin nhưng chỉ 75 % đúng")
DUNG = sach(75, nguon="95 % tự tin nhưng chỉ 75 % đúng")
print(f"ví dụ của chương: người rà soát chờ {100 - TU_TIN} % nhãn sơ bộ sai, thật ra sai {100 - DUNG} %,"
      f" gấp {(100 - DUNG) / (100 - TU_TIN):.0f} lần\n")

BIEN = np.linspace(0, 1, 11)


def hieu_chuan(p, y):
    tu_tin, dung = p.max(1), p.argmax(1) == y
    o = np.clip(np.digitize(tu_tin, BIEN[1:-1], right=True), 0, 9)
    bang = [(tu_tin[o == i].mean(), dung[o == i].mean(), np.mean(o == i)) for i in range(10) if np.any(o == i)]
    ece = sum(w * abs(a - c) for c, a, w in bang)
    return np.array(bang), ece, tu_tin, dung


HC = {}
for C in (1.0, 100.0):
    m = mo_hinh_al(thu_tu[:300], C=C)
    bang, ece, tu_tin, dung = hieu_chuan(m.predict_proba(Z_thu), y_thu)
    HC[C] = bang
    cao = tu_tin >= TU_TIN / 100
    print(f"C = {C:>5}: đúng {100 * dung.mean():.1f} %, ECE {ece:.3f};"
          f" {100 * cao.mean():.0f} % dự đoán tự tin từ 95 % trở lên, trong đó đúng {100 * dung[cao].mean():.1f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5))
ax.plot([0, 1], [0, 1], color="0.6", lw=1, ls="--")
ax.text(0.62, 0.66, "hiệu chuẩn hoàn hảo", rotation=38, fontsize=8, color="0.4")
for (C, bang), mau in zip(HC.items(), MAU):
    ax.plot(bang[:, 0], bang[:, 1], "o-", color=mau, lw=2, ms=6, label=f"C = {C:g}, 300 nhãn")
ax.plot([TU_TIN / 100], [DUNG / 100], "D", color=MAU[3], ms=8, zorder=3)
ax.annotate("ví dụ của chương:\n95 % tự tin, 75 % đúng", (TU_TIN / 100, DUNG / 100), xytext=(0.7, 0.12),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5"))
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_xlabel("độ tự tin trung bình trong khoảng")
ax.set_ylabel("tỉ lệ dự đoán đúng trong khoảng")
ax.set_title("Biểu đồ hiệu chuẩn trên 2,000 ảnh thử (đo trên máy này)", fontsize=10)
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Một điểm nằm dưới đường chéo là một khoảng mà mô hình tự tin hơn mức nó đúng. Đó đúng là kiểu hỏng của ví dụ
trong chương: nếu quy trình tự nhận mọi nhãn sơ bộ có độ tự tin từ 95 % trở lên, tỉ lệ nhãn sai lọt qua là 100 %
trừ tỉ lệ đúng của khoảng đó, không phải 5 %. Vì vậy trước khi dùng độ tự tin để chọn mẫu hay để bỏ qua bước rà
soát, hãy đo hiệu chuẩn trên một tập có nhãn chuẩn (ground truth), và theo dõi tỉ lệ người rà soát phải sửa nhãn
sơ bộ như chương đề nghị.

## Thử thêm

1. Ở phần 1, đặt `GIAY_MOI_CUA_SO = 0.5` (cửa sổ trượt nửa giây). Số cửa sổ mỗi tháng gấp đôi; FPR mỗi cửa sổ
   phải nhỏ đến đâu, và cần bao nhiêu ngày âm thanh để chứng minh nó?
2. Ở phần 4, in `sum(bo_nho(c40, byte_vong=1).values())`, tức bộ đệm vòng MFCC lưu dạng int8. Cấu hình 40 hệ
   số đã vừa 16 KB chưa? Nếu chưa, thứ gì còn chiếm nhiều nhất?
3. Ở phần 7, đặt `BUOC_AL, SO_VONG_AL = 400, 3`, để cùng mua tới 1,300 nhãn nhưng mỗi vòng mua nhiều hơn.
   Khoảng cách giữa hai chiến lược thay đổi thế nào?

## Tóm lại

* Với 2,592,000 cửa sổ mỗi tháng, một lần đánh thức sai mỗi tháng đòi FPR khoảng 3.9 × 10⁻⁷; "đúng 99 %" thành
  25,920 lần, nên KWS phải được đánh giá bằng số lần nhận sai mỗi giờ.
* Ngân sách $150K của napkin math 1.3 mua 750K nhãn, dưới mốc 1M; các mức tăng của bảng 3 không cộng thẳng được,
  và tiền nhãn trong kịch bản của chương gấp 520.8–1,562.5 lần một lượt huấn luyện.
* MFCC có kích thước do số khung và số hệ số quyết định, không do tần số lấy mẫu; số hệ số quyết định bộ đệm vòng,
  thứ làm cấu hình 40 hệ số vượt 16 KB trong bản kê của sổ tay.
* Một mô hình học trên dữ liệu sạch, hay chỉ trên một bộ sinh, có thể hỏng trên điều kiện và nhóm người mà nó chưa
  thấy; hỗn hợp dữ liệu phải được kiểm định trên dữ liệu đại diện cho lúc triển khai.
* Kappa từ 0.85 xuống 0.72 có thể nghĩa là tỉ lệ nhãn sai gần gấp đôi; học chủ động chỉ đáng tiền khi tiền chấm
  điểm nằm trong ngân sách và độ tự tin của mô hình đã được hiệu chuẩn.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật dữ liệu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch04-data-engineering/), dựng từ chương
[*Data Engineering*](https://mlsysbook.ai/vol1/data_engineering/data_engineering.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.